# Split Federated Learning V2 + Projected:
### CIFAR-10 + α = 0.1

In [1]:
# =============================================================================
# SFLV2 + Orthogonal Projection (Gated) on CIFAR-10 (Dirichlet α=0.1)
# - Warmup + alpha ramp + norm-preserving projection
# - Train-side feature reservoir (no test leakage)
# - Rebuild U every 3 rounds; clear buffer after rebuild
# - SGD + momentum + cosine LR (server); label smoothing
# - local_ep = 2; CIFAR-10 RandAug-lite; freeze client BN after warmup
# =============================================================================
import torch
from torch import nn
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import math
import os.path
import pandas as pd
from sklearn.model_selection import train_test_split
from PIL import Image
from glob import glob
from pandas import DataFrame
import random
import numpy as np
import os


import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import copy
import time

# ----------------------------- Repro & Device --------------------------------
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    print(torch.cuda.get_device_name(0))   
    

    
def _sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

#====================Paremeters Change================================
program = "SFLV2_Projected-CF10-01-100EP-5WP-3RB-0-05LR-4096FM"
DATASET_NAME = "CIFAR10"
ALPHA = 0.1     # Non-IID split


DATA_ROOT = "data"
print(f"---------{program}----------")              # this is to identify the program in the slurm outputs files


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pin = torch.cuda.is_available()


# To print in color -------test/train of the client side
def prRed(skk): print("\033[91m {}\033[00m" .format(skk)) 
def prGreen(skk): print("\033[92m {}\033[00m" .format(skk))     


#===================================================================
BATCH_SIZE = 256
LR_BASE = 0.05
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
LABEL_SMOOTH = 0.0
NUM_WORKERS = 4


# No. of users
NUM_CLIENTS = 10
EPOCHS = 100
LOCAL_EPOCHS = 5
FRAC = 1        # participation of clients; if 1 then 100% clients participate in SFLV2



#=====================================================================================================
#                           Orthogonal Config
#=====================================================================================================
# Orthogonal projection config deque
ORTH = True
TOP_R = 8 # Smaller r to avoid over-pruning
rebuild_every = 3         # rebuild subspace cadence (rounds)
WARMUP_ROUNDS = 5         # no projection before this
alpha_max = 0.1           # target projection strength
alpha_min = 0.0
alpha_warm_len = 10       # rounds to ramp alpha after warmup

# total samples for building U_old
# Feature reservoir (train-side only)
FEATURE_MEM_CAP = 4096    # train-side feature reservoir
FEATURE_MEM = []
_seen = 0


U_old = None              # D x r matrix of principal directions at the cut
ell_global = 0            # current round index (for alpha schedule)


# ====================== Confirm =======================================
cut_at = "Late-cut at Layer 6 + AvgPool"
prRed(f"\tData:\t\t{DATASET_NAME}\n\tCut:\t\t{cut_at}\n\tGlobal Rounds:\t{EPOCHS}\n\tClients:\t{NUM_CLIENTS}\n\tLocal Epochs:\t{LOCAL_EPOCHS}\n\tα:\t\t{ALPHA}\n\tWarm-up:\t{WARMUP_ROUNDS}\n\tRebuild:\t{rebuild_every}")


#=====================================================================================================
#                       Round 4 New: Projection Gate
#=====================================================================================================

def proj_gate_strength(round_idx: int):
    """Alpha schedule for projection."""
    if round_idx < WARMUP_ROUNDS:
        return 0.0
    t = min(1.0, (round_idx - WARMUP_ROUNDS) / max(1, alpha_warm_len))
    return alpha_min + t * (alpha_max - alpha_min)

#=====================================================================================================
#                           Projection Help
#=====================================================================================================
def project_off_subspace_gated(grad_batch: torch.Tensor, U: torch.Tensor, alpha: float, preserve_norm=True):
    """g' = g - alpha * P_U(g), optionally rescale to preserve per-sample norm."""
    if (U is None) or (alpha <= 0.0):
        return grad_batch
    B = grad_batch.shape[0]
    g = grad_batch.view(B, -1)              # [B, D]
    comp = (g @ U) @ U.T                    # [B, D]
    if preserve_norm:
        n0 = g.norm(dim=1, keepdim=True).clamp_min(1e-12)
    g2 = g - alpha * comp
    if preserve_norm:
        n1 = g2.norm(dim=1, keepdim=True).clamp_min(1e-12)
        g2 = g2 * (n0 / n1)
    return g2.view_as(grad_batch)


def add_to_reservoir(rows_cpu_fp16: torch.Tensor):
    """Standard reservoir sampling to keep a fixed-size unbiased buffer."""
    global _seen, FEATURE_MEM
    for r in rows_cpu_fp16:
        _seen += 1
        if len(FEATURE_MEM) < FEATURE_MEM_CAP:
            FEATURE_MEM.append(r)
        else:
            j = np.random.randint(0, _seen)
            if j < FEATURE_MEM_CAP:
                FEATURE_MEM[j] = r

                
def collect_cut_feats_from_train(fx_client: torch.Tensor):
    """Collect a small subset of cut features from TRAIN forward passes only."""
    with torch.no_grad():
        feats = fx_client.detach().view(fx_client.shape[0], -1)  # [B, D]
        k = min(24, feats.shape[0])
        idx = torch.randperm(feats.shape[0], device=feats.device)[:k]
        add_to_reservoir(feats[idx].cpu().to(torch.float16))


#=====================================================================================================
#         Subspace builder (simple PCA/SVD on buffered features)
#=====================================================================================================
def build_U_old_from_memory(top_r=TOP_R, device='cpu'):
    """Build principal subspace from the reservoir via PCA (pca_lowrank)."""
    if len(FEATURE_MEM) == 0:
        return None
    X = torch.stack(FEATURE_MEM).to(torch.float32)     # [N, D]
    X = X - X.mean(dim=0, keepdim=True)
    U, S, V = torch.pca_lowrank(X.cpu(), q=min(top_r, X.shape[1]), center=False)
    return V[:, :top_r].to(device, dtype=torch.float32)

def clear_feature_mem():
    global FEATURE_MEM, _seen
    FEATURE_MEM.clear()
    _seen = 0

    
#=====================================================================================================
#                           Baseblock
#=====================================================================================================
class Baseblock(nn.Module):
    expansion = 1
    def __init__(self, input_planes, planes, stride = 1, dim_change = None):
        # super(Baseblock, self).__init__()
        super().__init__()
        self.conv1 = nn.Conv2d(input_planes, planes, stride =  stride, kernel_size = 3, padding = 1)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, stride = 1, kernel_size = 3, padding = 1)
        self.bn2 = nn.BatchNorm2d(planes)
        self.dim_change = dim_change
        
        
    def forward(self, x):
        res = x
        output = F.relu(self.bn1(self.conv1(x)))
        output = self.bn2(self.conv2(output))
        
        if self.dim_change is not None:
            res =self.dim_change(res)
            
        output += res
        output = F.relu(output)
        
        return output
        
    
#=====================================================================================================
#                           Client-side Model definition
#=====================================================================================================
# Model at client side
class ResNet18_client_side(nn.Module):
    def __init__(self, block, num_layers):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),  # 3x3 s=1
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            # no maxpool for 32x32
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
        )

        # add layer 3-6 + avg-pool (copied from Client)
        self.layer3 = nn.Sequential (
                nn.Conv2d(64, 64, kernel_size = 3, stride = 1, padding = 1),
                nn.BatchNorm2d(64),
                nn.ReLU (inplace = True),
                nn.Conv2d(64, 64, kernel_size = 3, stride = 1, padding = 1),
                nn.BatchNorm2d(64),       
                )   


        # self.input_planes is only used inside _layer()
        self.input_planes = 64

        self.layer4 = self._layer(block, 128, num_layers[0], stride = 2)
        self.layer5 = self._layer(block, 256, num_layers[1], stride = 2)
        self.layer6 = self._layer(block, 512, num_layers[2], stride = 2)
        
        
        # self. averagePool = nn.AvgPool2d(kernel_size = 7, stride = 1)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        

        # Weights into loop (the same thing as SERVER)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
        

    # a helper function to build deeper ResNet blocks (layer4–6) using Baseblock
    def _layer(self, block, planes, num_layers, stride = 2):
        dim_change = None
        if stride != 1 or planes != self.input_planes * block.expansion:
            dim_change = nn.Sequential(nn.Conv2d(self.input_planes, planes*block.expansion, kernel_size = 1, stride = stride),
                                       nn.BatchNorm2d(planes*block.expansion))
        netLayers = []
        netLayers.append(block(self.input_planes, planes, stride = stride, dim_change = dim_change))
        self.input_planes = planes * block.expansion
        for i in range(1, num_layers):
            netLayers.append(block(self.input_planes, planes))
            self.input_planes = planes * block.expansion
            
        return nn.Sequential(*netLayers)

        
    def forward(self, x):
        resudial1 = F.relu(self.layer1(x))
        out1 = self.layer2(resudial1)
        out1 = out1 + resudial1 # adding the resudial inputs -- downsampling not required in this layer
        x = F.relu(out1)

        out2 = self.layer3(x)
        out2 = out2 + x
        x = F.relu(out2)

        x = self.layer4(x)                    # channels = 128
        x = self.layer5(x)                    # channels = 256
        x = self.layer6(x)                    # Channels = 512
        x = self.avgpool(x)                   # shape [B, 512, H', W']

        # MID-Cut: Send x to server
        return x            # [B, 512, H', W']


net_glob_client = ResNet18_client_side(Baseblock, [2, 2, 2])
if torch.cuda.device_count() > 1:
    print("We use",torch.cuda.device_count(), "GPUs")
    net_glob_client = nn.DataParallel(net_glob_client)   # to use the multiple GPUs; later we can change this to CPUs only 

net_glob_client.to(device)
print(net_glob_client)    


#=====================================================================================================
#                           Server-side Model definition
#=====================================================================================================
# Model at client side
class ResNet18_server_side(nn.Module):
    def __init__(self, block, classes):
        super(ResNet18_server_side, self).__init__()
        self.fc = nn.Linear(512 * block.expansion, classes)


        
        # Weights into loop
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
    
        
    
    def forward(self, x):
        # x = [B, 512, 1, 1] from the Client side
        x = x.view(x.size(0), -1)            # [B, 512] 
        y_hat =self.fc(x)                    # [B, NUM_CLASSES]
        
        return y_hat

    
# === INSTANTIATE MODELS ===
if DATASET_NAME == "CIFAR10":
    net_glob_server = ResNet18_server_side(Baseblock, 10)
else:
    net_glob_server = ResNet18_server_side(Baseblock, 100)
    

if torch.cuda.device_count() > 1:
    print("We use",torch.cuda.device_count(), "GPUs")
    net_glob_server = nn.DataParallel(net_glob_server)   # to use the multiple GPUs 


net_glob_server.to(device)
print(net_glob_server)  


#===================================================================================
# For Server Side Loss and Accuracy 
loss_train_collect = []
acc_train_collect = []
loss_test_collect = []
acc_test_collect = []
batch_acc_train = []
batch_loss_train = []
batch_acc_test = []
batch_loss_test = []
round_time_collect = [] # Stores seconds per global round

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)


# Persistent server optimizer & scheduler (like SL)
optimizer_server = torch.optim.SGD(net_glob_server.parameters(), lr=LR_BASE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
server_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_server, T_max=EPOCHS, eta_min=5e-4)


#====================================================================================================
#                                  Server Side Programs
#====================================================================================================
def calculate_accuracy(fx, y):
    preds = fx.max(1, keepdim=True)[1]
    correct = preds.eq(y.view_as(preds)).sum()
    acc = 100.00 *correct.float()/preds.shape[0]
    return acc

# to print train - test together in each round-- these are made global
acc_avg_all_user_train = 0
loss_avg_all_user_train = 0
loss_train_collect_user = []
acc_train_collect_user = []
loss_test_collect_user = []
acc_test_collect_user = []


#client idx collector
idx_collect = []
l_epoch_check = False
fed_check = False

count1 = 0
count2 = 0


# Server-side function associated with Training 
def train_server(fx_client, y, l_epoch_count, l_epoch, idx, len_batch):
    global net_glob_server, criterion, device
    global batch_acc_train, batch_loss_train, l_epoch_check, fed_check
    global loss_train_collect, acc_train_collect, count1
    global acc_avg_all_user_train, loss_avg_all_user_train, idx_collect
    global loss_train_collect_user, acc_train_collect_user
    global U_old, ell_global, optimizer_server  

    net_glob_server.train()
    # train and update
    optimizer_server.zero_grad()
    
    
    fx_client = fx_client.to(device)
    y = y.to(device)
    
    #---------forward prop-------------
    fx_server = net_glob_server(fx_client)
    
    # calculate loss
    loss = criterion(fx_server, y)
    # calculate accuracy
    acc = calculate_accuracy(fx_server, y)
    
    #--------backward prop--------------
    loss.backward()
    
    
    # ---- Projection on ∂L/∂f_c before returning to client ----
    dfx_client = fx_client.grad.clone().detach()
    if ORTH and (U_old is not None):
        alpha = proj_gate_strength(ell_global)
        dfx_client = project_off_subspace_gated(dfx_client, U_old.to(dfx_client.dtype), alpha, preserve_norm=True)
    # -----------------------------------------------------------
    
    
    optimizer_server.step()
    
    batch_loss_train.append(loss.item())
    batch_acc_train.append(acc.item())
    
    # server-side model net_glob_server is global so it is updated automatically in each pass to this function
    
    # count1: to track the completion of the local batch associated with one client
    count1 += 1
    if count1 == len_batch:
        acc_avg_train = sum(batch_acc_train)/len(batch_acc_train)           # it has accuracy for one batch
        loss_avg_train = sum(batch_loss_train)/len(batch_loss_train)
        
        batch_acc_train = []
        batch_loss_train = []
        count1 = 0
        
        prRed('Client{} Train => Local Epoch: {} \tAcc: {:.3f} \tLoss: {:.4f}'.format(idx, l_epoch_count, acc_avg_train, loss_avg_train))
        
                
        # If one local epoch is completed, after this a new client will come
        if l_epoch_count == l_epoch-1:
            
            l_epoch_check = True                # to evaluate_server function - to check local epoch has completed or not
                       
            # we store the last accuracy in the last batch of the epoch and it is not the average of all local epochs
            # this is because we work on the last trained model and its accuracy (not earlier cases)
            
            #print("accuracy = ", acc_avg_train)
            acc_avg_train_all = acc_avg_train
            loss_avg_train_all = loss_avg_train
                        
            # accumulate accuracy and loss for each new user
            loss_train_collect_user.append(loss_avg_train_all)
            acc_train_collect_user.append(acc_avg_train_all)
            
            # collect the id of each new user                        
            if idx not in idx_collect:
                idx_collect.append(idx) 
                #print(idx_collect)
        
        # This is to check if all users are served for one round --------------------
        if len(idx_collect) == NUM_CLIENTS:
            fed_check = True                                                  # to evaluate_server function  - to check fed check has hitted
            # all users served for one round ------------------------- output print and update is done in evaluate_server()
            # for nicer display 
                        
            idx_collect = []
            
            acc_avg_all_user_train = sum(acc_train_collect_user)/len(acc_train_collect_user)
            loss_avg_all_user_train = sum(loss_train_collect_user)/len(loss_train_collect_user)
            
            loss_train_collect.append(loss_avg_all_user_train)
            acc_train_collect.append(acc_avg_all_user_train)
            
            acc_train_collect_user = []
            loss_train_collect_user = []
            
    # send gradients to the client               
    return dfx_client

# Server-side functions associated with Testing
def evaluate_server(fx_client, y, idx, len_batch, ell):
    global net_glob_server, criterion, batch_acc_test, batch_loss_test
    global loss_test_collect, acc_test_collect, count2, NUM_CLIENTS, acc_avg_train_all, loss_avg_train_all, l_epoch_check, fed_check
    global loss_test_collect_user, acc_test_collect_user, acc_avg_all_user_train, loss_avg_all_user_train
    
    net_glob_server.eval()
  
    with torch.no_grad():
        fx_client = fx_client.to(device)
        y = y.to(device) 
        #---------forward prop-------------
        fx_server = net_glob_server(fx_client)
        
        # calculate loss
        loss = criterion(fx_server, y)
        # calculate accuracy
        acc = calculate_accuracy(fx_server, y)
        
        
        batch_loss_test.append(loss.item())
        batch_acc_test.append(acc.item())
        
               
        count2 += 1
        if count2 == len_batch:
            acc_avg_test = sum(batch_acc_test)/len(batch_acc_test)
            loss_avg_test = sum(batch_loss_test)/len(batch_loss_test)
            
            batch_acc_test = []
            batch_loss_test = []
            count2 = 0
            
            prGreen('Client{} Test =>                   \tAcc: {:.3f} \tLoss: {:.4f}'.format(idx, acc_avg_test, loss_avg_test))
            
            # if a local epoch is completed   
            if l_epoch_check:
                l_epoch_check = False
                
                # Store the last accuracy and loss
                acc_avg_test_all = acc_avg_test
                loss_avg_test_all = loss_avg_test
                        
                loss_test_collect_user.append(loss_avg_test_all)
                acc_test_collect_user.append(acc_avg_test_all)
                
            # if all users are served for one round ----------                    
            if fed_check:
                fed_check = False
                                
                acc_avg_all_user = sum(acc_test_collect_user)/len(acc_test_collect_user)
                loss_avg_all_user = sum(loss_test_collect_user)/len(loss_test_collect_user)
            
                loss_test_collect.append(loss_avg_all_user)
                acc_test_collect.append(acc_avg_all_user)
                acc_test_collect_user = []
                loss_test_collect_user= []
                              
                print("====================== SERVER V1==========================")
                print(' Train: Round {:3d}, Avg Accuracy {:.3f} | Avg Loss {:.3f}'.format(ell, acc_avg_all_user_train, loss_avg_all_user_train))
                print(' Test: Round {:3d}, Avg Accuracy {:.3f} | Avg Loss {:.3f}'.format(ell, acc_avg_all_user, loss_avg_all_user))
                # print("==========================================================")
                
                
            # Rebuild U on cadence (no feature collection here)
            if ORTH and (ell % rebuild_every == 1):
                global U_old
                U_old = build_U_old_from_memory(top_r=TOP_R, device=device)
                clear_feature_mem()
         
    return 


# ----------------------------- FedAvg (client) -------------------------------
def FedAvg(w_list):
    w_avg = copy.deepcopy(w_list[0])
    for k in w_avg.keys():
        for i in range(1, len(w_list)):
            w_avg[k] += w_list[i][k]
        w_avg[k] = torch.div(w_avg[k], len(w_list))
    return w_avg

#=============================================================================
#                         Data loading 
#============================================================================= 
from torchvision import datasets, transforms

# CIFAR-10 & CIFAR-100 transforms
cifar_mean = {
    "CIFAR10":  [0.4914, 0.4822, 0.4465],
    "CIFAR100": [0.5071, 0.4867, 0.4408],
}[DATASET_NAME]
cifar_std = {
    "CIFAR10":  [0.2470, 0.2435, 0.2616],
    "CIFAR100": [0.2675, 0.2565, 0.2761],
}[DATASET_NAME]

# Try RandAugment if available
try:
    from torchvision.transforms import RandAugment
    ra = [RandAugment(num_ops=2, magnitude=9)]
except Exception:
    ra = []

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
] + ra + [
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

if DATASET_NAME == "CIFAR10":
    dataset_train = datasets.CIFAR10(root=DATA_ROOT, train=True,  download=True, transform=train_transforms)
    dataset_test  = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=test_transforms)
else:
    dataset_train = datasets.CIFAR100(root=DATA_ROOT, train=True,  download=True, transform=train_transforms)
    dataset_test  = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=test_transforms)

print("Train samples:", len(dataset_train), " Test samples:", len(dataset_test))
print("Example train label:", dataset_train.targets[0])

# ----------------------------- Partitions -----------------------------------
class DatasetSplit(Dataset):
    def __init__(self, dataset, idxs):
        self.dataset = dataset; self.idxs = list(idxs)
    def __len__(self): return len(self.idxs)
    def __getitem__(self, item): return self.dataset[self.idxs[item]]

def dataset_iid(dataset, NUM_CLIENTS):
    num_items = int(len(dataset)/NUM_CLIENTS)
    dict_users, all_idxs = {}, [i for i in range(len(dataset))]
    for i in range(NUM_CLIENTS):
        dict_users[i] = set(np.random.choice(all_idxs, num_items, replace=False))
        all_idxs = list(set(all_idxs) - dict_users[i])
    return dict_users

def dataset_noniid_dirichlet(dataset, NUM_CLIENTS, alpha=ALPHA, seed=SEED):
    rng = np.random.default_rng(seed)
    labels = np.array(dataset.targets)
    n_classes = len(np.unique(labels))
    dict_users = {i: set() for i in range(NUM_CLIENTS)}
    for c in range(n_classes):
        idxs = np.where(labels == c)[0]
        rng.shuffle(idxs)
        p = rng.dirichlet([alpha]*NUM_CLIENTS)
        cuts = (np.cumsum(p)*len(idxs)).astype(int)[:-1]
        splits = np.split(idxs, cuts)
        for client_id, split in enumerate(splits):
            dict_users[client_id].update(split.tolist())
    return dict_users

dict_users      = dataset_noniid_dirichlet(dataset_train, NUM_CLIENTS, alpha=ALPHA, seed=SEED)
dict_users_test = dataset_iid(dataset_test, NUM_CLIENTS)


#=============================================================================
#                         Forgetting metric setup 
#============================================================================= 

from collections import defaultdict

# Treat each global round as a window (quick baseline).
# If you want true distribution shifts, predefine your own windows and fill `val_loaders`.
T = EPOCHS

val_loaders = {}
# Deterministic, disjoint slices of the test set: window t gets items t, t+T, t+2T, ...
rng_idx = np.arange(len(dataset_test))
for t in range(T):
    idxs = rng_idx[t::T]
    val_loaders[t] = DataLoader(
        DatasetSplit(
            dataset_test, 
            idxs), 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin
    )

best_loss_per_window = defaultdict(lambda: float('inf'))  # stores min historical loss per window
forgetting_log = []  # per round dicts: {"round": t, "mean_forgetting": x, "per_window": {...}}
# ============================================================================

def _eval_loss_on_loader(model_client, model_server, loader, device, criterion):
    """Evaluate mean loss on a loader using split models."""
    # Handle DataParallel transparently
    mc = model_client.module if isinstance(model_client, nn.DataParallel) else model_client
    ms = model_server.module if isinstance(model_server, nn.DataParallel) else model_server
    mc.eval(); ms.eval()
    loss_sum, n = 0.0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            fx = mc(images)
            logits = ms(fx)
            loss = criterion(logits, labels)
            bs = labels.size(0)
            loss_sum += loss.item() * bs
            n += bs
    return loss_sum / max(1, n)


# ------------------------------- Client --------------------------------------
def freeze_client_bn(net):
    for m in net.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

class Client(object):
    def __init__(self, net_client_model, idx, lr, device,
                 dataset_train=None, dataset_test=None, idxs=None, idxs_test=None):
        self.idx = idx
        self.device = device
        self.lr = LR_BASE
        self.local_ep = LOCAL_EPOCHS
        self.ldr_train = DataLoader(
            DatasetSplit(
                dataset_train, 
                idxs), 
            batch_size=BATCH_SIZE, 
            shuffle=True, 
            drop_last=False,
            num_workers=NUM_WORKERS,
            pin_memory=pin
        )
        self.ldr_test  = DataLoader(
            DatasetSplit(
                dataset_test,  
                idxs_test), 
            batch_size=BATCH_SIZE, 
            shuffle=False, 
            drop_last=False,
            num_workers=NUM_WORKERS,
            pin_memory=pin
        )

    def train(self, net):
        net.train()
        optimizer_client = torch.optim.SGD(net.parameters(), lr=self.lr, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
        for ep in range(self.local_ep):
            len_batch = len(self.ldr_train)
            for batch_idx, (images, labels) in enumerate(self.ldr_train):
                images, labels = images.to(self.device), labels.to(self.device)
                optimizer_client.zero_grad()
                fx = net(images)
                # collect train-side cut features into reservoir
                if ORTH:
                    collect_cut_feats_from_train(fx)
                client_fx = fx.clone().detach().requires_grad_(True)
                dfx = train_server(client_fx, labels, ep, self.local_ep, self.idx, len_batch)
                fx.backward(dfx)
                optimizer_client.step()
        return net.state_dict()

    def evaluate(self, net, ell):
        net.eval()
        with torch.no_grad():
            len_batch = len(self.ldr_test)
            for batch_idx, (images, labels) in enumerate(self.ldr_test):
                images, labels = images.to(self.device), labels.to(self.device)
                fx = net(images)
                evaluate_server(fx, labels, self.idx, len_batch, ell)
        return

# ------------------------------- Training ------------------------------------
net_glob_client.train()
w_glob_client = net_glob_client.state_dict()  # initial global client weights

acc_train_collect.clear(); loss_train_collect.clear()
acc_test_collect.clear();  loss_test_collect.clear()
round_time_collect.clear()

for iter in range(EPOCHS):
    ell_global = iter

    # Freeze BN (not timed)
    if iter == WARMUP_ROUNDS:
        if isinstance(net_glob_client, nn.DataParallel):
            freeze_client_bn(net_glob_client.module)
        else:
            freeze_client_bn(net_glob_client)

    # ---- START TIMER (training+testing+FedAvg only) ----
    _sync_cuda()
    t_start = time.perf_counter()
    # ----------------------------------------------------

    m = max(int(FRAC * NUM_CLIENTS), 1)
    idxs_users = np.random.choice(range(NUM_CLIENTS), m, replace=False)

    w_locals_client = []
    for idx in idxs_users:
        local = Client(net_glob_client, idx, LR_BASE, device,
                       dataset_train=dataset_train, dataset_test=dataset_test,
                       idxs=dict_users[idx], idxs_test=dict_users_test[idx])
        w_client = local.train(net=copy.deepcopy(net_glob_client).to(device))
        w_locals_client.append(copy.deepcopy(w_client))
        local.evaluate(net=copy.deepcopy(net_glob_client).to(device), ell=iter)

    # FedAvg
    w_glob_client = FedAvg(w_locals_client)
    net_glob_client.load_state_dict(w_glob_client)

    # ---- STOP TIMER & LOG ----
    _sync_cuda()
    dt = time.perf_counter() - t_start
    round_time_collect.append(dt)
    # --------------------------

    # === Forgetting metric update at end of round (not counted in time) ===
    t = iter
    L_cur_t = _eval_loss_on_loader(net_glob_client, net_glob_server, val_loaders[t], device, criterion)
    best_loss_per_window[t] = min(best_loss_per_window[t], L_cur_t)

    per_win_fgt = {}
    for t_prev in range(t):
        L_now = _eval_loss_on_loader(net_glob_client, net_glob_server, val_loaders[t_prev], device, criterion)
        Fgt = L_now - best_loss_per_window[t_prev]
        per_win_fgt[t_prev] = Fgt

    mean_fgt = float(np.mean(list(per_win_fgt.values()))) if per_win_fgt else 0.0
    forgetting_log.append({"round": t, "mean_forgetting": mean_fgt, "per_window": per_win_fgt})
    print(f"Forgetting : Round {t}, mean_forgetting : {mean_fgt:.3f}")
    print("==========================================================")


    # LR schedule (also outside timing)
    server_sched.step()

print("Training and Evaluation completed!")

# ----------------------------- Save outputs ----------------------------------
n = min(len(acc_train_collect), len(acc_test_collect), len(round_time_collect), len(forgetting_log))
if n == 0:
    # Safeguard: if per-round aggregates weren’t populated (e.g., single-user path),
    # still produce a minimal log so the files exist.
    n = len(round_time_collect)
round_process = list(range(1, n+1))
mean_fgt_series = [d["mean_forgetting"] for d in forgetting_log[:n]] if forgetting_log else [0.0]*n

df = DataFrame({
    'round': round_process,
    'acc_train': acc_train_collect[:n],
    'acc_test':  acc_test_collect[:n],
    'round_time_sec': round_time_collect[:n],
    'mean_FgT_loss': mean_fgt_series,
})

file_name_csv  = program + ".csv"
df.to_csv(file_name_csv, index=False)
print("Saved:", file_name_csv)

#=============================================================================
#                         Program Completed
#============================================================================= 

NVIDIA H200
---------SFLV2 + Projected - CIFAR-10 α=0.1----------
 	Data:		CIFAR10
	Cut:		Late-cut at Layer 6 + AvgPool
	Global Rounds:	100
	Clients:	10
	Local Epochs:	5
	α:		0.1
	Warm-up:	5
	Rebuild:	3
ResNet18_client_side(
  (layer1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layer2): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (layer3): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1,